# Agentic RAG Using Langraph

In [15]:
import os
import time
from dotenv import load_dotenv
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from huggingface_hub import get_collection
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from google import genai
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient
load_dotenv()
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq
from langchain_community.document_loaders import PyPDFLoader

In [16]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2998.16it/s]


In [17]:
parser = StrOutputParser()

In [18]:
#all api keys

#QDRANT
QDRANT_API_KEY = os.getenv("QDRANTAPIKEY")
QDRANT_ENDPOINT = os.getenv("QDRANTENDPOINT")

#gemini model apikey
geminiapikey = os.getenv("GEMINIAPIKEY")

#
# llm = ChatGoogleGenerativeAI(
#     model="gemini-2.5-flash-lite",
#     google_api_key=geminiapikey,
#     temperature=0
# )


groq_api_key = os.getenv("GROQAPIKEY")  # make sure this matches your .env variable name

llm = ChatGroq(
    model="qwen/qwen3-32b",
    api_key=groq_api_key,
    temperature=0
)

In [19]:
start = time.time()
try:
    test = llm.invoke("Say hello in one word.")
    print("SUCCESS:", test)
except Exception as e:
    print("FAILED:", e)
print(f"Took {time.time() - start:.1f}s")

SUCCESS: content='<think>\nOkay, the user wants me to say hello in one word. Let me think about the possible options. The most straightforward is "Hello" itself, but maybe they want a different approach. Words like "Hi" or "Hey" are shorter, but "Hello" is the most direct. Are there any other single words that convey greeting? Maybe "Greetings" but that\'s a bit longer. "Salutations" is another option but it\'s more formal. The user probably expects a simple and common greeting. Let me confirm if there\'s any cultural context or specific language they might be referring to. Since they didn\'t specify, sticking with the standard English greeting makes sense. So the best answer is "Hello".\n</think>\n\nHello.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 151, 'prompt_tokens': 14, 'total_tokens': 165, 'completion_time': 0.320964703, 'completion_tokens_details': None, 'prompt_time': 0.00061908, 'prompt_tokens_details': None, 'queue_time': 0.007061854, 'total

In [20]:
file_path = "who.pdf"
loader = PyPDFLoader(file_path)

document = loader.load()

Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 11 0 (offset 0)
Ignoring wrong pointing object 13 0 (offset 0)
Ignoring wrong pointing object 15 0 (offset 0)
Ignoring wrong pointing object 21 0 (offset 0)
Ignoring wrong pointing object 38 0 (offset 0)


In [21]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=130,
    separators=[
        "\n\n",   # paragraphs (highest priority)
        "\n",     # lines
        ". ",     # sentences
        " ",      # words
        ""        # fallback
    ]
)

chunks = splitter.split_documents(document)

In [22]:
client = QdrantClient(
    url=QDRANT_ENDPOINT,
    api_key=QDRANT_API_KEY
)

collections = client.get_collections().collections

collection_exists = any(
    collection.name == "agentic_rag"
    for collection in collections
)

# =========================
# 7. Create Collection Only If Not Exists
# =========================

if not collection_exists:

    vectorstore = QdrantVectorStore.from_documents(
        documents=chunks,
        embedding=embeddings,
        url=QDRANT_ENDPOINT,
        api_key=QDRANT_API_KEY,
        collection_name="agentic_rag",
    )

    print("Collection created and documents stored.")

else:

    vectorstore = QdrantVectorStore.from_existing_collection(
        embedding=embeddings,
        url=QDRANT_ENDPOINT,
        api_key=QDRANT_API_KEY,
        collection_name="agentic_rag",
    )

    print("Collection already exists. Using existing collection.")

C:\Users\Personal\PycharmProjects\NLP-RAG-BY-HOPETOSKILL\.venv\Lib\site-packages\qdrant_client\qdrant_remote.py:290: UserWarning: Failed to obtain server version. Unable to check client-server compatibility. Set check_compatibility=False to skip version check.
  show_warning(


Collection created and documents stored.


In [23]:
from typing import TypedDict, List, Optional


In [24]:

class AgentState(TypedDict):
    """State schema for our Agentic RAG system.

    This is the shared clipboard that every node can read and write to.
    """
    question: str
    documents: List[str]
    answer: str
    iterations: int
    search_type: str   #vectorstore or web search


In [31]:
def retrieve(state: AgentState)->dict:
    """Retrieve the relevant documents from the database"""

    print(f"  [RETRIEVE] Searching for: {state['question'][:50]}...")

    docs = vectorstore.invoke(state["question"])

    docs_text = [doc.page_content for doc in docs]

    print(f"  [RETRIEVE] Found {len(docs_text)} documents")

    return {
        "documents": docs_text,
        "search_type": "vectorstore",
    }



In [30]:
grade_prompt = ChatPromptTemplate.from_template(
    """You are a document relevance grader.
Determine if the following document is relevant to the question.
Reply with ONLY 'yes' or 'no'.

Document: {document}

Question: {question}

Is this document relevant? (yes/no):"""
)

In [ ]:
def grade_documents(state: AgentState)->dict:

    relevnt_docs = []

    grade_chain = grade_prompt | llm | StrOutputParser()



